# Fase 0 — Baseline em Alta Precisão: Seleção e Validação do Teacher FP16/BF16

**Notebook:** `00_baseline_teacher.ipynb`  
**Fase:** 0 de 4  
**Projeto:** Arquitetura Híbrida Multimodal em torno de bitnet.cpp  

---

## Resumo

Este notebook implementa a **Fase 0** do pipeline de implementação descrito na especificação técnica (Seção 5.1). O objetivo desta fase é estabelecer o modelo *teacher* em alta precisão (FP16/BF16), validar o alinhamento entre encoder e tokenizer, fixar as interfaces de modalidade e definir os critérios formais de aceite para as fases subsequentes.

O modelo *teacher* constitui o ponto de partida para a transição gradual 16-bit → 1.58-bit implementada na Fase 3. A literatura de *continual quantization-aware pre-training* demonstra que essa rota é superior ao treinamento integral em baixa precisão e que a retenção do estado do otimizador ao longo das fases reduz *spikes* de perda.

---

## Índice

1. [Instalação de Dependências](#1-instalação-de-dependências)
2. [Configuração Global](#2-configuração-global)
3. [Montagem do Google Drive](#3-montagem-do-google-drive)
4. [Fundamentação Teórica](#4-fundamentação-teórica)
5. [Seleção e Carregamento do Teacher](#5-seleção-e-carregamento-do-teacher)
6. [Validação: Encoder e Tokenizer](#6-validação-encoder-e-tokenizer)
7. [Definição das Interfaces de Modalidade](#7-definição-das-interfaces-de-modalidade)
8. [Definição dos Critérios de Aceite](#8-definição-dos-critérios-de-aceite)
9. [Avaliação Baseline do Teacher](#9-avaliação-baseline-do-teacher)
10. [Persistência dos Artefatos](#10-persistência-dos-artefatos)
11. [Conclusões e Próximos Passos](#11-conclusões-e-próximos-passos)

## 1. Instalação de Dependências

As versões são fixadas explicitamente para garantir reprodutibilidade entre sessões do Colab.

In [ ]:
# Instalar dependências com versões fixadas — suprimir output para manter o notebook limpo
!pip install -q \
    transformers==4.44.0 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    evaluate==0.4.2 \
    sentencepiece==0.2.0 \
    protobuf==4.25.4 \
    safetensors==0.4.3 \
    einops==0.8.0

print("Instalação concluída.")

## 2. Configuração Global

Todas as constantes do experimento são definidas nesta célula. Seeds são fixadas de forma determinística para garantir reprodutibilidade total do pipeline.

In [ ]:
import logging
import os
import random
import sys

import numpy as np
import torch

# ---------------------------------------------------------------------------
# Configuração de logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("phase0")

# ---------------------------------------------------------------------------
# Seeds de reprodutibilidade (obrigatórias por especificação — Seção 0.3)
# ---------------------------------------------------------------------------
SEED: int = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---------------------------------------------------------------------------
# Dispositivo de computação
# ---------------------------------------------------------------------------
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Dispositivo de computação: %s", DEVICE)
if DEVICE.type == "cuda":
    logger.info(
        "GPU: %s | VRAM disponível: %.2f GiB",
        torch.cuda.get_device_name(0),
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
    )

# ---------------------------------------------------------------------------
# Constantes do experimento
# ---------------------------------------------------------------------------
TEACHER_MODEL_ID: str = "microsoft/bitnet-b1.58-2B-4T"  # Substitua pelo teacher FP16 desejado
DTYPE_TEACHER: torch.dtype = torch.bfloat16             # Precisão do teacher
MAX_SEQ_LEN: int = 2048                                  # Comprimento máximo de contexto
EVAL_BATCH_SIZE: int = 4                                 # Tamanho de batch para avaliação
DRIVE_PROJECT_DIR: str = "/content/drive/MyDrive/multimodal-ternary-llm"
PHASE_NAME: str = "phase0_baseline"

logger.info("Configuração global inicializada. Seed=%d | dtype=%s", SEED, DTYPE_TEACHER)

## 3. Montagem do Google Drive

O Google Drive é utilizado para persistência de checkpoints, logs e métricas entre sessões do Colab, conforme especificado na Seção 0.3.

In [ ]:
from pathlib import Path

# Montar o Google Drive (ignorar silenciosamente se fora do Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    logger.info("Google Drive montado em /content/drive")
except ImportError:
    logger.warning("Ambiente não-Colab detectado. Saltando montagem do Drive.")

# Criar estrutura de diretórios
CHECKPOINT_DIR = Path(DRIVE_PROJECT_DIR) / "checkpoints" / PHASE_NAME
LOG_DIR        = Path(DRIVE_PROJECT_DIR) / "logs"
METRICS_DIR    = Path(DRIVE_PROJECT_DIR) / "metrics"

for d in (CHECKPOINT_DIR, LOG_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)
    logger.info("Diretório assegurado: %s", d)

## 4. Fundamentação Teórica

### 4.1 Papel do Modelo Teacher

O modelo *teacher* em FP16/BF16 desempenha dois papéis críticos no pipeline:

1. **Referência de desempenho** — Estabelece a linha de base contra a qual todas as versões quantizadas serão comparadas ao longo das fases subsequentes (critérios de aceite, Seção 7 da especificação).
2. **Fonte de supervisão por destilação** — Os logits e as representações intermediárias do *teacher* são utilizados como sinal de treinamento nas Fases 2 e 3 via *KL divergence distillation loss* e *representation alignment loss*.

### 4.2 Estratégia de Transição de Precisão

A literatura de *continual quantization-aware pre-training* demonstra que a rota 16-bit → 1.58-bit é superior ao treinamento integral em baixa precisão. A Fase 0 fixa o ponto de partida dessa trajetória, garantindo que o *teacher* seja validado antes de qualquer operação de quantização.

### 4.3 Seleção do Modelo Teacher

Para fins de prototipagem, emprega-se o modelo `microsoft/bitnet-b1.58-2B-4T` em modo de carregamento FP16, conforme disponível no HuggingFace Hub. Em contextos de produção, o *teacher* deve ser o modelo de referência de maior capacidade disponível na família arquitetural selecionada.

## 5. Seleção e Carregamento do Teacher

O modelo *teacher* é carregado na precisão alvo (BF16) com descarregamento automático para CPU em caso de restrição de VRAM.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

logger.info("Carregando tokenizer: %s", TEACHER_MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL_ID,
    use_fast=True,
    trust_remote_code=True,
)
# Garantir token de padding para compatibilidade com batches
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    logger.info("pad_token definido como eos_token: %r", tokenizer.eos_token)

logger.info("Carregando modelo teacher: %s | dtype=%s", TEACHER_MODEL_ID, DTYPE_TEACHER)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    torch_dtype=DTYPE_TEACHER,
    device_map="auto",       # Distribui automaticamente entre GPU e CPU
    trust_remote_code=True,
)
teacher_model.eval()

# Congela o teacher — inferência apenas, sem atualização de gradientes
for param in teacher_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in teacher_model.parameters())
logger.info("Teacher carregado. Total de parâmetros: %s", f"{total_params:,}")

## 6. Validação: Encoder e Tokenizer

Esta célula verifica a coerência entre o tokenizer e o modelo, e confirma que as dimensões relevantes estão corretamente configuradas para integração com o connector e o backbone BitNet nas fases subsequentes.

In [ ]:
from typing import Dict, Any

# ---------------------------------------------------------------------------
# Validação das dimensões do modelo
# ---------------------------------------------------------------------------
config = teacher_model.config
D_MODEL: int = config.hidden_size
VOCAB_SIZE: int = config.vocab_size
N_LAYERS: int = config.num_hidden_layers
N_HEADS: int = config.num_attention_heads

logger.info("Configuração do teacher:")
logger.info("  d_model    = %d", D_MODEL)
logger.info("  vocab_size = %d", VOCAB_SIZE)
logger.info("  n_layers   = %d", N_LAYERS)
logger.info("  n_heads    = %d", N_HEADS)

# ---------------------------------------------------------------------------
# Verificação de alinhamento tokenizer ↔ modelo
# ---------------------------------------------------------------------------
assert tokenizer.vocab_size <= VOCAB_SIZE, (
    f"Inconsistência: tokenizer.vocab_size ({tokenizer.vocab_size}) "
    f"> model.vocab_size ({VOCAB_SIZE})."
)
logger.info("Alinhamento tokenizer ↔ modelo: OK")

# ---------------------------------------------------------------------------
# Forward pass de sanidade
# ---------------------------------------------------------------------------
SAMPLE_PROMPT: str = "The architecture of large language models has evolved significantly"
inputs = tokenizer(SAMPLE_PROMPT, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = teacher_model(**inputs)

assert outputs.logits.shape[-1] == VOCAB_SIZE, (
    f"Dimensão de saída inesperada: {outputs.logits.shape[-1]} ≠ {VOCAB_SIZE}."
)
logger.info(
    "Forward pass de sanidade concluído. Logits shape: %s",
    tuple(outputs.logits.shape),
)

print(f"\n{'='*60}")
print(f"  Teacher válido: d_model={D_MODEL}, vocab={VOCAB_SIZE}, layers={N_LAYERS}")
print(f"{'='*60}\n")

## 7. Definição das Interfaces de Modalidade

Esta célula registra formalmente as dimensões de interface esperadas para cada módulo de percepção, de acordo com a Seção 3.2 da especificação técnica. Estas constantes serão reutilizadas pelos notebooks das Fases 1–3.

In [ ]:
import json

# ---------------------------------------------------------------------------
# Registro das interfaces de modalidade
# Ajuste os valores de d_enc conforme os encoders selecionados
# ---------------------------------------------------------------------------
MODALITY_INTERFACES: Dict[str, Any] = {
    "text": {
        "description": "Tokenização padrão do backbone — saída como embeddings d_model.",
        "d_enc": D_MODEL,
        "d_model": D_MODEL,
        "connector_required": False,
    },
    "vision": {
        "description": "ViT encoder com resolução dinâmica (e.g., Qwen2.5-VL ViT).",
        "d_enc": 1152,   # Dimensão típica do ViT-L / ViT empregado em Qwen2.5-VL
        "d_model": D_MODEL,
        "connector_required": True,
        "pooling": "token_pooling",
        "temporal_encoding": True,
    },
    "timeseries": {
        "description": "Encoder Mamba / state space model para séries temporais e sensores.",
        "d_enc": 256,    # Ajustar conforme o encoder SSM selecionado
        "d_model": D_MODEL,
        "connector_required": True,
        "optional": True,
    },
}

# Persistir as interfaces como artefato JSON para referência nas fases seguintes
interfaces_path = METRICS_DIR / "modality_interfaces.json"
with open(interfaces_path, "w", encoding="utf-8") as f:
    json.dump(MODALITY_INTERFACES, f, indent=2, ensure_ascii=False)

logger.info("Interfaces de modalidade registradas em: %s", interfaces_path)
print(json.dumps(MODALITY_INTERFACES, indent=2, ensure_ascii=False))

## 8. Definição dos Critérios de Aceite

Os critérios de aceite formalizam as condições sob as quais cada fase transicional pode ser considerada bem-sucedida, em conformidade com a Seção 7 da especificação técnica.

In [ ]:
# ---------------------------------------------------------------------------
# Critérios de aceite formalizados (Seção 7 da especificação)
# ---------------------------------------------------------------------------
ACCEPTANCE_CRITERIA: Dict[str, Any] = {
    "max_perplexity_degradation_pct": 10.0,   # Máximo de degradação de perplexidade aceitável
    "max_downstream_degradation_pct": 5.0,    # Degradação máx. em benchmarks downstream
    "latency_reduction_required": True,        # Inferência ternária deve ser mais rápida
    "logit_collapse_threshold": 1e-4,          # Desvio padrão mínimo dos logits para detectar colapso
    "memory_reduction_required": True,         # Backbone ternário deve usar menos memória
    "multimodal": {
        "visual_parsing_preserved": True,      # Capacidade de parsing visual não deve colapsar
        "ocr_preserved": True,                 # OCR estrutural deve ser mantido
    },
}

criteria_path = METRICS_DIR / "acceptance_criteria.json"
with open(criteria_path, "w", encoding="utf-8") as f:
    json.dump(ACCEPTANCE_CRITERIA, f, indent=2)

logger.info("Critérios de aceite persistidos em: %s", criteria_path)
print(json.dumps(ACCEPTANCE_CRITERIA, indent=2))

## 9. Avaliação Baseline do Teacher

Computa a perplexidade do *teacher* em um conjunto de validação representativo. Este valor constitui o **baseline de referência** contra o qual todos os modelos quantizados das fases subsequentes serão comparados.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import math

# ---------------------------------------------------------------------------
# Dataset de validação (WikiText-2 como proxy padrão; ajustar conforme domínio)
# ---------------------------------------------------------------------------
logger.info("Carregando dataset de validação: wikitext-2-raw-v1")
val_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")

def tokenise_batch(examples: Dict[str, Any]) -> Dict[str, Any]:
    """
    Tokenise a batch of text examples for language modelling.

    Parameters
    ----------
    examples : dict
        HuggingFace dataset batch containing a 'text' key.

    Returns
    -------
    dict
        Tokenised batch with 'input_ids' and 'attention_mask'.
    """
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        return_tensors=None,
    )

tokenised_val = val_dataset.map(
    tokenise_batch,
    batched=True,
    remove_columns=val_dataset.column_names,
)
tokenised_val.set_format(type="torch")

# ---------------------------------------------------------------------------
# Cálculo da perplexidade
# ---------------------------------------------------------------------------
def compute_perplexity(
    model: torch.nn.Module,
    dataset,
    batch_size: int = EVAL_BATCH_SIZE,
    max_batches: int = 50,
    device: torch.device = DEVICE,
) -> float:
    """
    Compute the perplexity of a causal language model on a given dataset.

    Parameters
    ----------
    model : torch.nn.Module
        Causal language model in evaluation mode.
    dataset : datasets.Dataset
        Tokenised dataset with 'input_ids' and 'attention_mask'.
    batch_size : int
        Evaluation batch size.
    max_batches : int
        Maximum number of batches to evaluate (for time constraints on Colab).
    device : torch.device
        Compute device.

    Returns
    -------
    float
        Perplexity score.
    """
    loader = DataLoader(dataset, batch_size=batch_size)
    model.eval()
    total_nll: float = 0.0
    total_tokens: int = 0

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = input_ids.clone()
            # Mascarar tokens de padding na loss
            labels[attention_mask == 0] = -100

            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_nll += out.loss.item() * n_tokens
            total_tokens += n_tokens

    ppl = math.exp(total_nll / max(total_tokens, 1))
    return ppl

logger.info("Calculando perplexidade baseline do teacher...")
teacher_perplexity: float = compute_perplexity(teacher_model, tokenised_val)
logger.info("Perplexidade baseline (teacher FP16/BF16): %.4f", teacher_perplexity)
print(f"\n  Perplexidade Baseline: {teacher_perplexity:.4f}\n")

## 10. Persistência dos Artefatos

Os artefatos desta fase — métricas baseline, interfaces de modalidade e critérios de aceite — são persistidos no Google Drive para utilização nas fases subsequentes.

In [ ]:
# ---------------------------------------------------------------------------
# Persistência das métricas baseline
# ---------------------------------------------------------------------------
baseline_metrics: Dict[str, Any] = {
    "phase": "phase0_baseline",
    "teacher_model_id": TEACHER_MODEL_ID,
    "dtype": str(DTYPE_TEACHER),
    "perplexity_wikitext2": teacher_perplexity,
    "d_model": D_MODEL,
    "vocab_size": VOCAB_SIZE,
    "n_layers": N_LAYERS,
    "seed": SEED,
}

metrics_path = METRICS_DIR / "phase0_baseline_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2)

logger.info("Métricas baseline persistidas em: %s", metrics_path)

# ---------------------------------------------------------------------------
# Persistência do estado do tokenizer (reutilizado pelas fases seguintes)
# ---------------------------------------------------------------------------
tokenizer_save_path = CHECKPOINT_DIR / "tokenizer"
tokenizer.save_pretrained(str(tokenizer_save_path))
logger.info("Tokenizer salvo em: %s", tokenizer_save_path)

print("\nArtefatos da Fase 0 persistidos com sucesso.")
print(json.dumps(baseline_metrics, indent=2))

## 11. Conclusões e Próximos Passos

### Resultados da Fase 0

A Fase 0 concluiu com sucesso a validação do modelo *teacher* FP16/BF16. Os seguintes artefatos foram produzidos e persistidos:

| Artefato | Localização |
|---|---|
| Métricas baseline | `metrics/phase0_baseline_metrics.json` |
| Interfaces de modalidade | `metrics/modality_interfaces.json` |
| Critérios de aceite | `metrics/acceptance_criteria.json` |
| Tokenizer | `checkpoints/phase0_baseline/tokenizer/` |

### Próxima Fase

Prosseguir para o notebook `01_connector_pretraining.ipynb` (**Fase 1**), que implementa o pré-treinamento do conector MLP com encoders e backbone congelados, conforme a Seção 5.2 da especificação técnica.

> **Condição de prosseguimento:** A perplexidade baseline obtida nesta fase deve ser registrada como referência formal. Qualquer modelo produzido nas fases subsequentes com degradação superior a `max_perplexity_degradation_pct` deve ser revisado antes de avançar.